Read satellite data


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio

GIMMS_PHENOLOGY_DIR = "../../data/satellite_data/images/PKU-GIMMS/phenology"
GIMMS_PHENOLOGY_PATTERN = "GIMMS_Phenology_SnowFilter_Forest1114_{year}.tif"

def load_gimms_snowfilter_sos_eos(df, years, phenology_dir=GIMMS_PHENOLOGY_DIR):
        coords = list(zip(df["longitude"].values, df["latitude"].values))
    n = len(coords)
    sample_year = next(
        y for y in years
        if os.path.exists(os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=y)))
    )
    with rasterio.open(os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=sample_year))) as src:
        transform = src.transform
        height, width = src.height, src.width

    rows = np.empty(n, dtype=np.int32)
    cols = np.empty(n, dtype=np.int32)
    for i, (lon, lat) in enumerate(coords):
        r, c = rasterio.transform.rowcol(transform, lon, lat)
        rows[i], cols[i] = r, c

    valid_rc = (rows >= 0) & (cols >= 0) & (rows < height) & (cols < width)

    for year in years:
        fp = os.path.join(phenology_dir, GIMMS_PHENOLOGY_PATTERN.format(year=year))
        if not os.path.exists(fp):
            print(f"  missing phenology file: {fp}", flush=True)
            df[f"sos_{year}"] = np.nan
            df[f"eos_{year}"] = np.nan
            continue
        with rasterio.open(fp) as src:
            sos_band = src.read(1)
            eos_band = src.read(2)
        sos = np.full(n, np.nan, dtype=np.float32)
        eos = np.full(n, np.nan, dtype=np.float32)
        sos[valid_rc] = sos_band[rows[valid_rc], cols[valid_rc]]
        eos[valid_rc] = eos_band[rows[valid_rc], cols[valid_rc]]
        sos[~np.isfinite(sos)] = np.nan
        eos[~np.isfinite(eos)] = np.nan
        df[f"sos_{year}"] = sos
        df[f"eos_{year}"] = eos
    return df

def _load_annual_climate_for_coords(lons, lats, years):
        key = set(zip(np.round(lons, 5), np.round(lats, 5)))
    years = set(int(y) for y in years)
    coords = pd.DataFrame({"longitude": lons, "latitude": lats})
    coords["_k"] = list(zip(np.round(coords["longitude"], 5), np.round(coords["latitude"], 5)))

    def load_chunked(paths, prefix):
        hits = []
        for fp in paths:
            cols = pd.read_csv(fp, nrows=0).columns
            ycols = [c for c in cols if c.startswith(prefix) and int(c.split("_")[-1]) in years]
            if not ycols:
                continue
            usecols = ["longitude", "latitude"] + ycols
            for chunk in pd.read_csv(fp, usecols=usecols, chunksize=200_000):
                mask = [
                    (round(lo, 5), round(la, 5)) in key
                    for lo, la in zip(chunk["longitude"], chunk["latitude"])
                ]
                sub = chunk.loc[mask]
                if len(sub):
                    hits.append(sub)
        if not hits:
            return coords[["longitude", "latitude"]].copy()
        df = pd.concat(hits, ignore_index=True)
        df["_k"] = list(zip(np.round(df["longitude"], 5), np.round(df["latitude"], 5)))
        ycols = [c for c in df.columns if c.startswith(prefix)]
        df = df.groupby("_k", as_index=False)[ycols].first()
        return coords[["_k"]].merge(df, on="_k", how="left")

    t_paths = [
        "../../data/climate_data/tables/climate_data/temp/temp-1982-1999.csv",
        "../../data/climate_data/tables/climate_data/temp/temp-2000-2024.csv",
    ]
    p_paths = [
        "../../data/climate_data/tables/climate_data/prcp/prcp-1982-1999.csv",
        "../../data/climate_data/tables/climate_data/prcp/prcp-2000-2024.csv",
    ]
    print("  loading annual T from climate tables...", flush=True)
    tdf = load_chunked(t_paths, "annual_t_")
    print("  loading annual P from climate tables...", flush=True)
    pdf = load_chunked(p_paths, "annual_p_")
    out = coords[["longitude", "latitude"]].copy()
    for c in tdf.columns:
        if c.startswith("annual_t_"):
            out[c] = tdf[c].values
    for c in pdf.columns:
        if c.startswith("annual_p_"):
            out[c] = pdf[c].values
    return out

def read_satellite_data(veg_type, satellite):
    veg_class = pd.read_csv("../../data/veg_class_data/tables/veg_class.csv")
    clim_fp = f"../../data/satellite_data/tables/phenology_climate/{satellite}.csv"
    if satellite == "gimms" and not os.path.exists(clim_fp):
        print(f"  {clim_fp} missing — rebuilding annual T/P from climate tables", flush=True)
        forest = veg_class[veg_class["veg_class"].isin([12, 13, 14])].copy()
        if veg_type in (12, 13, 14):
            forest = forest[forest["veg_class"] == veg_type].copy()
        years = list(range(1982, 2023))
        df_satellite = _load_annual_climate_for_coords(
            forest["longitude"].values, forest["latitude"].values, years
        )
        df = forest.merge(df_satellite, on=["longitude", "latitude"], how="inner")
    else:
        df_satellite = pd.read_csv(clim_fp)
        df = pd.merge(df_satellite, veg_class, on=["latitude", "longitude"], how="inner")

    if veg_type in (12, 13, 14):
        df = df[df["veg_class"].isin([veg_type])]
    else:
        df = df[df["veg_class"].isin([12, 13, 14])]

    eos_cols = [col for col in df.columns if "eos" in col]
    t_cols = [col for col in df.columns if "annual_t" in col]
    p_cols = [col for col in df.columns if "annual_p" in col]
    sos_cols = [col for col in df.columns if "sos" in col]

    if satellite == "gimms":
        years = [str(y) for y in range(1982, 2023)]
        df = df.drop(columns=[c for c in eos_cols + sos_cols], errors="ignore")
        df = load_gimms_snowfilter_sos_eos(df, [int(y) for y in years])
        eos_cols = [col for col in df.columns if col.startswith("eos_")]
        sos_cols = [col for col in df.columns if col.startswith("sos_")]
    elif satellite == "modis":
        years = [str(y) for y in range(2001, 2024)]
        mask_sos = (df[sos_cols] < 0).any(axis=1)
        mask_eos = (df[eos_cols] > 365).any(axis=1)
        df = df[~(mask_sos | mask_eos)].copy()
    else:
        years = [str(y) for y in range(2013, 2023)]
        mask_sos = (df[sos_cols] < 0).any(axis=1)
        mask_eos = (df[eos_cols] > 365).any(axis=1)
        df = df[~(mask_sos | mask_eos)].copy()

    cols = years
    df = df[[col for col in eos_cols + t_cols + p_cols + sos_cols if any(y in col for y in cols)] + ["latitude", "longitude", "veg_class"]].copy()
    t_cols_df = [col for col in df.columns if "annual_t" in col]
    df[t_cols_df] = df[t_cols_df] - 273.5  # Convert temperature
    df.columns = df.columns.str.replace(r"\D*(\d{4})$", lambda m: f"{m.group(0)[0:-4]}{m.group(1)}", regex=True)
    df["annual_t"] = df[[col for col in df.columns if "annual_t" in col]].mean(axis=1)
    df["annual_p"] = df[[col for col in df.columns if "annual_p" in col]].mean(axis=1)
    eos_year_cols = [col for col in df.columns if col.startswith("eos_")]
    sos_year_cols = [col for col in df.columns if col.startswith("sos_")]
    df["eos"] = df[eos_year_cols].mean(axis=1)
    df["sos"] = df[sos_year_cols].mean(axis=1)
    if satellite == "gimms":
        df = df[df["eos"].notna()].copy()
    return df


Plot heatmap


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.ticker import FormatStrFormatter

def filter_extremes(df, x_col, y_col, z_col):
    x_all = df[x_col].values
    y_all = df[y_col].values*1000
    z_all = df[z_col].values
    x_low, x_high = np.nanpercentile(x_all, [0, 100])
    y_low, y_high = np.nanpercentile(y_all, [0, 100])
    mask = (x_all >= x_low) & (x_all <= x_high) & (y_all >= y_low) & (y_all <= y_high)
    return x_all[mask], y_all[mask], z_all[mask]

def classify_data(x, y, satellite):
    temp_class = np.where(x < 7.25, 'low temp', 'High temp')
    if satellite == 'gimms':
        precip_class = np.where(y < 1000, 'low precip', 'High precip')
    elif satellite == 'modis':
        precip_class = np.where(y < 1000, 'low precip', 'High precip')
    else:
        precip_class = np.where(y < 1100, 'low precip', 'High precip')
    return temp_class, precip_class

def compute_binned_means(x, y, z, min_z_count):
    x_bins = np.arange(-7, 20.5, 0.5)
    y_bins = np.arange(300, 1850, 100)
    z_mean = np.zeros((len(x_bins)-1, len(y_bins)-1)) * np.nan
    z_count = np.zeros((len(x_bins)-1, len(y_bins)-1))

    for i in range(len(x_bins)-1):
        for j in range(len(y_bins)-1):
            mask = (x >= x_bins[i]) & (x < x_bins[i+1]) & (y >= y_bins[j]) & (y < y_bins[j+1])
            if np.sum(mask) >= min_z_count:
                z_mean[i, j] = np.nanmean(z[mask])
                z_count[i, j] = np.sum(mask)

    x_step = x_bins[1] - x_bins[0]
    y_step = y_bins[1] - y_bins[0]
    x_edges = np.concatenate([[x_bins[0] - x_step / 2], x_bins + x_step / 2])
    y_edges = np.concatenate([[y_bins[0] - y_step / 2], y_bins + y_step / 2])
    x_centers = (x_edges[:-1] + x_edges[1:]) / 2
    y_centers = (y_edges[:-1] + y_edges[1:]) / 2

    print(f'Max count of grid: {np.nanmax(z_count)}')
    z_mean_masked = z_mean
    print(np.nanmin(z_mean_masked))
    return z_mean_masked, x_centers, y_centers

def plot_main_heatmap(x, y, z_mean, name1, name2, x_bins, y_bins, x_display_range, y_display_range, vlim, satellite, breakpoint_temp=None, breakpoint_temp_wet=None):
    ax = plt.subplot2grid((6, 4), (0, 1), rowspan=3, colspan=2)
    cmap = LinearSegmentedColormap.from_list(
        "custom_cmap",
        ["#0b3c68", "#165188", "#2066a8", "#4d91c4", "#8ec1da", 
         "#fbebe1", "#f6d6c2", "#d47264", "#c14d48", "#ae282c"]
    )
    vmin, vmax = vlim
    norm = Normalize(vmin=vmin if vmin is not None else np.nanmin(z_mean),
                     vmax=vmax if vmax is not None else np.nanmax(z_mean))
    x_step = x_bins[1] - x_bins[0]
    y_step = y_bins[1] - y_bins[0]
    x_edges = np.concatenate([[x_bins[0] - x_step / 2], x_bins + x_step / 2])
    y_edges = np.concatenate([[y_bins[0] - y_step / 2], y_bins + y_step / 2])

    ax.imshow(z_mean.T, origin='lower',
              extent=[x_edges[0], x_edges[-1], y_edges[0], y_edges[-1]],
              cmap=cmap, norm=norm, aspect='auto')

    ax.set_xlim(*x_display_range)
    ax.set_ylim(*y_display_range)
    ax.set_yticks([400, 800, 1200, 1600])
    ax.set_ylabel("MAP (mm/year)", fontsize=14)
    ax.set_xticklabels([])
    plt.xticks(fontsize=12)
    plt.yticks(fontsize=12)
    if satellite == 'gimms':
        p_thresh = 1000
        t_break = breakpoint_temp if breakpoint_temp is not None and np.isfinite(breakpoint_temp) else 7.25
        t_break_wet = breakpoint_temp_wet if breakpoint_temp_wet is not None and np.isfinite(breakpoint_temp_wet) else 7.25
    elif satellite == 'modis':
        p_thresh = 1000
        t_break = breakpoint_temp if breakpoint_temp is not None and np.isfinite(breakpoint_temp) else 6.75
        t_break_wet = breakpoint_temp_wet if breakpoint_temp_wet is not None and np.isfinite(breakpoint_temp_wet) else 6.75
    else:
        p_thresh = 1100
        t_break = breakpoint_temp if breakpoint_temp is not None and np.isfinite(breakpoint_temp) else 7.25
        t_break_wet = breakpoint_temp_wet if breakpoint_temp_wet is not None and np.isfinite(breakpoint_temp_wet) else 7.25

    ax.axhline(y=p_thresh, color='black', linestyle='--', linewidth=1.0)
    if t_break is not None and np.isfinite(t_break):
        ax.vlines(x=t_break, ymin=ax.get_ylim()[0], ymax=p_thresh,
                  colors='black', linestyles='--', linewidth=1.0)

    ax.text(x_display_range[0] + 0.5,
            y_display_range[1] - 50,
            "Wet regions",
            ha='left', va='top', fontsize=10, color='black')
    ax.text(x_display_range[0] + 0.5,
            y_display_range[0] + 50,
            "Cold-dry regions",
            ha='left', va='bottom', fontsize=10, color='black')
    ax.text(x_display_range[1] - 0.5,
            y_display_range[0] + 50,
            "Hot-dry regions",
            ha='right', va='bottom', fontsize=10, color='black')
    return ax, norm

def add_colorbar(fig, norm, z_col, vlim):
    cmap = LinearSegmentedColormap.from_list(
        "custom_cmap",
        ["#0b3c68", "#165188", "#2066a8", "#4d91c4", "#8ec1da", 
         "#fbebe1", "#f6d6c2", "#d47264", "#c14d48", "#ae282c"]
    )
    cbar_ax = fig.add_axes([0.65, 0.22, 0.018, 0.5]) ###############################################################
    cbar = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cbar_ax, extend='neither')
    vmin, vmax = vlim
    import numpy as np
    ticks = np.arange(vmin + 10, vmax, 10).tolist()
    cbar.set_ticks(ticks)
    cbar.ax.yaxis.set_major_formatter(FormatStrFormatter('%.0f'))
    if z_col == "eos":
        cbar.ax.set_ylabel("EOS (DOY)", fontsize=14)
    elif z_col == "trend_eos":
        cbar.ax.set_ylabel("EOS trend (DOY/year)", fontsize=14)
    cbar.ax.tick_params(labelsize=12)

def _dry_breakpoint_from_raw_grids(x, y, z, satellite, min_seg=20):
    from scipy.stats import linregress
    if satellite == 'gimms':
        thresh, dlim1, dlim2 = 1000, -6.5, 12
    elif satellite == 'modis':
        thresh, dlim1, dlim2 = 1000, -7.5, 13
    else:
        thresh, dlim1, dlim2 = 1100, -7.5, 13.5

    dry = (y < thresh) & np.isfinite(x) & np.isfinite(z) & (x >= dlim1) & (x <= dlim2)
    x_dry = np.asarray(x[dry], dtype=float)
    z_dry = np.asarray(z[dry], dtype=float)
    def bin_to_half_degree(vals):
        return np.floor(vals * 2) / 2 + 0.25
    x_binned = bin_to_half_degree(x_dry)
    bin_mids = np.unique(np.sort(x_binned))
    mids, means = [], []
    for xm in bin_mids:
        mask = x_binned == xm
        if np.any(mask):
            mids.append(xm)
            means.append(np.nanmean(z_dry[mask]))
    mids = np.asarray(mids, dtype=float)
    means = np.asarray(means, dtype=float)
    if len(mids) == 0:
        return mids, means, np.nan
    peak_mat = float(mids[int(np.nanargmax(means))])
    candidates = mids[mids >= peak_mat]
    best_sse, best_bp = np.inf, np.nan
    for bp in candidates:
        left = (x_dry <= bp)
        right = (x_dry >= bp)
        if left.sum() < min_seg or right.sum() < min_seg:
            continue
        s1, i1, *_ = linregress(x_dry[left], z_dry[left])
        s2, i2, *_ = linregress(x_dry[right], z_dry[right])
        pred = np.empty_like(z_dry)
        pred[left] = s1 * x_dry[left] + i1
        pred[right] = s2 * x_dry[right] + i2
        both = left & right
        if both.any():
            pred[both] = 0.5 * ((s1 * x_dry[both] + i1) + (s2 * x_dry[both] + i2))
        sse = np.sum((z_dry - pred) ** 2)
        if sse < best_sse:
            best_sse, best_bp = sse, float(bp)

    if not np.isnan(best_bp):
        from scipy.stats import f
        left = (x_dry <= best_bp)
        right = (x_dry >= best_bp)
        s1, i1, *_ = linregress(x_dry[left], z_dry[left])
        s2, i2, *_ = linregress(x_dry[right], z_dry[right])
        pred = np.empty_like(z_dry)
        pred[left] = s1 * x_dry[left] + i1
        pred[right] = s2 * x_dry[right] + i2
        both = left & right
        if both.any():
            pred[both] = 0.5 * ((s1 * x_dry[both] + i1) + (s2 * x_dry[both] + i2))
        sse_p = np.sum((z_dry - pred) ** 2)
        s_s, i_s, *_ = linregress(x_dry, z_dry)
        pred_s = s_s * x_dry + i_s
        sse_s = np.sum((z_dry - pred_s) ** 2)
        N = len(x_dry)
        df2 = N - 4
        if df2 > 0:
            F = ((sse_s - sse_p) / 2) / (sse_p / df2)
            p_val = 1 - f.cdf(F, 2, df2)
            sse_improvement = (sse_s - sse_p) / sse_s
            if p_val < 0.05 and sse_improvement >= 0.015:
                return mids, means, best_bp
    return mids, means, np.nan

def _wet_breakpoint_from_raw_grids(x, y, z, satellite, min_seg=20):
    from scipy.stats import linregress
    if satellite == 'gimms':
        thresh, wlim1, wlim2 = 1000, -4.5, 20
    elif satellite == 'modis':
        thresh, wlim1, wlim2 = 1000, -3.5, 20
    else:
        thresh, wlim1, wlim2 = 1100, -3.5, 20

    wet = (y >= thresh) & np.isfinite(x) & np.isfinite(z) & (x >= wlim1) & (x <= wlim2)
    x_wet = np.asarray(x[wet], dtype=float)
    z_wet = np.asarray(z[wet], dtype=float)
    def bin_to_half_degree(vals):
        return np.floor(vals * 2) / 2 + 0.25
    x_binned = bin_to_half_degree(x_wet)
    bin_mids = np.unique(np.sort(x_binned))
    mids, means = [], []
    for xm in bin_mids:
        mask = x_binned == xm
        if np.any(mask):
            mids.append(xm)
            means.append(np.nanmean(z_wet[mask]))
    mids = np.asarray(mids, dtype=float)
    means = np.asarray(means, dtype=float)
    if len(mids) == 0:
        return mids, means, np.nan
    peak_mat = float(mids[int(np.nanargmax(means))])
    candidates = mids[mids >= peak_mat]
    best_sse, best_bp = np.inf, np.nan
    for bp in candidates:
        left = (x_wet <= bp)
        right = (x_wet >= bp)
        if left.sum() < min_seg or right.sum() < min_seg:
            continue
        s1, i1, *_ = linregress(x_wet[left], z_wet[left])
        s2, i2, *_ = linregress(x_wet[right], z_wet[right])
        pred = np.empty_like(z_wet)
        pred[left] = s1 * x_wet[left] + i1
        pred[right] = s2 * x_wet[right] + i2
        both = left & right
        if both.any():
            pred[both] = 0.5 * ((s1 * x_wet[both] + i1) + (s2 * x_wet[both] + i2))
        sse = np.sum((z_wet - pred) ** 2)
        if sse < best_sse:
            best_sse, best_bp = sse, float(bp)

    if not np.isnan(best_bp):
        from scipy.stats import f
        left = (x_wet <= best_bp)
        right = (x_wet >= best_bp)
        s1, i1, *_ = linregress(x_wet[left], z_wet[left])
        s2, i2, *_ = linregress(x_wet[right], z_wet[right])
        pred = np.empty_like(z_wet)
        pred[left] = s1 * x_wet[left] + i1
        pred[right] = s2 * x_wet[right] + i2
        both = left & right
        if both.any():
            pred[both] = 0.5 * ((s1 * x_wet[both] + i1) + (s2 * x_wet[both] + i2))
        sse_p = np.sum((z_wet - pred) ** 2)
        s_s, i_s, *_ = linregress(x_wet, z_wet)
        pred_s = s_s * x_wet + i_s
        sse_s = np.sum((z_wet - pred_s) ** 2)
        N = len(x_wet)
        df2 = N - 4
        if df2 > 0:
            F = ((sse_s - sse_p) / 2) / (sse_p / df2)
            p_val = 1 - f.cdf(F, 2, df2)
            sse_improvement = (sse_s - sse_p) / sse_s
            if p_val < 0.05 and sse_improvement >= 0.015:
                return mids, means, best_bp
    return mids, means, np.nan

def plot_heat_map(df, x_col, y_col, z_col, name1, name2, name3, title, vlim, min_z_count, satellite):
    x_display_range = (-7, 20)
    y_display_range = (200, 1750)
    x, y, z = filter_extremes(df, x_col, y_col, z_col)
    temp_class, precip_class = classify_data(x, y, satellite)
    z_mean, x_bins, y_bins = compute_binned_means(x, y, z, min_z_count)
    mid_x = (x_bins[:-1] + x_bins[1:]) / 2
    mid_y = (y_bins[:-1] + y_bins[1:]) / 2
    _, _, breakpoint_temp = _dry_breakpoint_from_raw_grids(x, y, z, satellite)
    _, _, breakpoint_temp_wet = _wet_breakpoint_from_raw_grids(x, y, z, satellite)
    fig = plt.figure(figsize=(12, 6))
    ax_main, norm = plot_main_heatmap(
        x, y, z_mean, name1, name2, x_bins, y_bins, x_display_range, y_display_range, vlim, satellite,
        breakpoint_temp=breakpoint_temp,
        breakpoint_temp_wet=breakpoint_temp_wet,
    )
    ax2, _ = plot_bottom_subplot(x, y, z_mean, x_bins, y_bins, precip_class, mid_x, x_display_range, z, vlim, satellite)
    add_colorbar(fig, norm, z_col, vlim)
    fig.subplots_adjust(left=0.08, right=0.80, top=0.92, bottom=0.08, hspace=0.7, wspace=0.3)
    plt.show()
    return fig

def plot_grid(df, name1, name2, name3, title, vlim, min_z_count, satellite):
    return plot_heat_map(df, name1, name2, name3, name1, name2, name3, title, vlim, min_z_count, satellite)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

def plot_bottom_subplot(x, y, z_mean, x_bins, y_bins, precip_class, mid_x, x_display_range, z, vlim, satellite):
    ax = plt.subplot2grid((8, 4), (4, 1), colspan=2, rowspan=3)
    if satellite == 'gimms':
        thresh = 1000
    elif satellite == 'modis':
        thresh = 1000
    else:
        thresh = 1100
    low_mask = y < thresh
    high_mask = y >= thresh

    def bin_to_half_degree(vals):
        return np.floor(vals * 2) / 2 + 0.25
    x_binned = bin_to_half_degree(x)
    bin_mids = np.unique(np.sort(x_binned))

    def binned_stats(x_vals, z_vals, bins, x_min, x_max):
        means, stds, mids = [], [], []
        for xm in bins:
            if xm < x_min or xm > x_max:
                continue
            mask = x_vals == xm
            if np.any(mask):
                means.append(np.nanmean(z_vals[mask]))
                stds.append(np.nanstd(z_vals[mask]))
                mids.append(xm)
        return np.array(mids), np.array(means), np.array(stds)

    if satellite == 'gimms':
        pos11, pos12 = 0.1, 0.5
        pos21, pos22 = 0.7, 0.2
        dlim1, dlim2 = -6.5, 12
        wlim1, wlim2 = -4.5, 20
    elif satellite == 'modis':
        pos11, pos12 = 0.1, 0.3
        pos21, pos22 = 0.7, 0.2
        dlim1, dlim2 = -7.5, 13
        wlim1, wlim2 = -3.5, 20
    else:
        pos11, pos12 = 0.1, 0.25
        pos21, pos22 = 0.7, 0.2
        dlim1, dlim2 = -7.5, 13.5
        wlim1, wlim2 = -3.5, 20
    mids_low, z_low_mean, z_low_std = binned_stats(x_binned[low_mask], z[low_mask], bin_mids, dlim1, dlim2)
    mids_high, z_high_mean, z_high_std = binned_stats(x_binned[high_mask], z[high_mask], bin_mids, wlim1, wlim2)

    ax.fill_between(mids_high, z_high_mean - z_high_std, z_high_mean + z_high_std,
                    color='gray', alpha=0.15, zorder=1)
    ax.plot(mids_high, z_high_mean, color='gray', lw=2, alpha=0.6, label="High prcp", zorder=1)

    ax.fill_between(mids_low, z_low_mean - z_low_std, z_low_mean + z_low_std,
                    color='#e03c31', alpha=0.15, zorder=2)
    ax.plot(mids_low, z_low_mean, color='#e03c31', lw=2, alpha=0.6, label="Low prcp", zorder=3)

    # Dry Breakpoint
    peak_mat = np.nan
    if len(mids_low) and np.any(np.isfinite(z_low_mean)):
        peak_mat = float(mids_low[int(np.nanargmax(z_low_mean))])
    x_dry = x[low_mask]
    z_dry = z[low_mask]
    in_range = np.isfinite(x_dry) & np.isfinite(z_dry) & (x_dry >= dlim1) & (x_dry <= dlim2)
    x_dry, z_dry = x_dry[in_range], z_dry[in_range]
    candidates = mids_low[np.isfinite(mids_low) & (mids_low >= peak_mat)] if np.isfinite(peak_mat) else mids_low
    best_sse, breakpoint_temp = np.inf, np.nan
    min_seg = 20
    for bp in candidates:
        left = x_dry <= bp
        right = x_dry >= bp
        if left.sum() < min_seg or right.sum() < min_seg:
            continue
        s1, i1, *_ = linregress(x_dry[left], z_dry[left])
        s2, i2, *_ = linregress(x_dry[right], z_dry[right])
        pred = np.empty_like(z_dry)
        pred[left] = s1 * x_dry[left] + i1
        pred[right] = s2 * x_dry[right] + i2
        both = left & right
        if both.any():
            pred[both] = 0.5 * ((s1 * x_dry[both] + i1) + (s2 * x_dry[both] + i2))
        sse = np.sum((z_dry - pred) ** 2)
        if sse < best_sse:
            best_sse, breakpoint_temp = sse, float(bp)

    dry_has_bp = False
    if not np.isnan(breakpoint_temp):
        from scipy.stats import f
        left = x_dry <= breakpoint_temp
        right = x_dry >= breakpoint_temp
        s1, i1, *_ = linregress(x_dry[left], z_dry[left])
        s2, i2, *_ = linregress(x_dry[right], z_dry[right])
        pred_p = np.empty_like(z_dry)
        pred_p[left] = s1 * x_dry[left] + i1
        pred_p[right] = s2 * x_dry[right] + i2
        both = left & right
        if both.any():
            pred_p[both] = 0.5 * ((s1 * x_dry[both] + i1) + (s2 * x_dry[both] + i2))
        sse_p = np.sum((z_dry - pred_p) ** 2)

        s_s, i_s, *_ = linregress(x_dry, z_dry)
        pred_s = s_s * x_dry + i_s
        sse_s = np.sum((z_dry - pred_s) ** 2)

        N = len(x_dry)
        df2 = N - 4
        if df2 > 0:
            F = ((sse_s - sse_p) / 2) / (sse_p / df2)
            p_val = 1 - f.cdf(F, 2, df2)
            sse_improvement = (sse_s - sse_p) / sse_s
            if p_val < 0.05 and sse_improvement >= 0.015:
                dry_has_bp = True

    # Wet Breakpoint
    peak_mat_high = np.nan
    if len(mids_high) and np.any(np.isfinite(z_high_mean)):
        peak_mat_high = float(mids_high[int(np.nanargmax(z_high_mean))])
    x_wet = x[high_mask]
    z_wet = z[high_mask]
    in_range_wet = np.isfinite(x_wet) & np.isfinite(z_wet) & (x_wet >= wlim1) & (x_wet <= wlim2)
    x_wet, z_wet = x_wet[in_range_wet], z_wet[in_range_wet]
    candidates_wet = mids_high[np.isfinite(mids_high) & (mids_high >= peak_mat_high)] if np.isfinite(peak_mat_high) else mids_high
    best_sse_wet, breakpoint_temp_wet = np.inf, np.nan
    for bp in candidates_wet:
        left = x_wet <= bp
        right = x_wet >= bp
        if left.sum() < min_seg or right.sum() < min_seg:
            continue
        s1, i1, *_ = linregress(x_wet[left], z_wet[left])
        s2, i2, *_ = linregress(x_wet[right], z_wet[right])
        pred = np.empty_like(z_wet)
        pred[left] = s1 * x_wet[left] + i1
        pred[right] = s2 * x_wet[right] + i2
        both = left & right
        if both.any():
            pred[both] = 0.5 * ((s1 * x_wet[both] + i1) + (s2 * x_wet[both] + i2))
        sse = np.sum((z_wet - pred) ** 2)
        if sse < best_sse_wet:
            best_sse_wet, breakpoint_temp_wet = sse, float(bp)

    wet_has_bp = False
    if not np.isnan(breakpoint_temp_wet):
        from scipy.stats import f
        left = x_wet <= breakpoint_temp_wet
        right = x_wet >= breakpoint_temp_wet
        s1, i1, *_ = linregress(x_wet[left], z_wet[left])
        s2, i2, *_ = linregress(x_wet[right], z_wet[right])
        pred_p = np.empty_like(z_wet)
        pred_p[left] = s1 * x_wet[left] + i1
        pred_p[right] = s2 * x_wet[right] + i2
        both = left & right
        if both.any():
            pred_p[both] = 0.5 * ((s1 * x_wet[both] + i1) + (s2 * x_wet[both] + i2))
        sse_p = np.sum((z_wet - pred_p) ** 2)

        s_s, i_s, *_ = linregress(x_wet, z_wet)
        pred_s = s_s * x_wet + i_s
        sse_s = np.sum((z_wet - pred_s) ** 2)

        N = len(x_wet)
        df2 = N - 4
        if df2 > 0:
            F = ((sse_s - sse_p) / 2) / (sse_p / df2)
            p_val = 1 - f.cdf(F, 2, df2)
            sse_improvement = (sse_s - sse_p) / sse_s
            if p_val < 0.05 and sse_improvement >= 0.015:
                wet_has_bp = True

    def format_p(p):
        if p < 0.01: return "p<0.01"
        elif p < 0.05: return "p<0.05"
        else: return f"p={p:.2f}"

    if not np.isnan(breakpoint_temp) and dry_has_bp:
        valid = np.isfinite(mids_low) & np.isfinite(z_low_mean)
        x_valid, z_valid = mids_low[valid], z_low_mean[valid]
        mask_below = x_valid <= breakpoint_temp
        mask_above = x_valid >= breakpoint_temp
        if np.sum(mask_below) >= 2:
            slope1, intercept1, r1, p1, _ = linregress(x_valid[mask_below], z_valid[mask_below])
            x_fit = np.linspace(x_valid[mask_below].min(), x_valid[mask_below].max(), 100)
            ax.plot(x_fit, slope1 * x_fit + intercept1, linestyle='--', color='#e4542a', linewidth=2.5, zorder=4)
            ax.text(pos11, pos12, f"R²={r1**2:.2f}, {format_p(p1)}", transform=ax.transAxes, fontsize=10, color="#e03c31", ha="left", va="bottom", zorder=5)
        if np.sum(mask_above) >= 2:
            slope2, intercept2, r2, p2, _ = linregress(x_valid[mask_above], z_valid[mask_above])
            x_fit = np.linspace(x_valid[mask_above].min(), x_valid[mask_above].max(), 100)
            ax.plot(x_fit, slope2 * x_fit + intercept2, linestyle='--', color='#e4542a', linewidth=2.5, zorder=4)
            ax.text(pos21, pos22, f"R²={r2**2:.2f}, {format_p(p2)}", transform=ax.transAxes, fontsize=10, color="#e03c31", ha="left", va="bottom", zorder=5)
        ax.axvline(breakpoint_temp, color='#e4542a', linestyle='-', linewidth=2, alpha=0.5, zorder=4)
        ax.text(breakpoint_temp + 2, ax.get_ylim()[1] - 5, f"{breakpoint_temp:.2f}°C", color="#e4542a", fontsize=10, ha="center", va="top", weight="bold", zorder=5)
    else:
        valid = np.isfinite(mids_low) & np.isfinite(z_low_mean)
        x_valid, z_valid = mids_low[valid], z_low_mean[valid]
        slope, intercept, r_val, p_val, _ = linregress(x_valid, z_valid)
        x_fit = np.linspace(x_valid.min(), x_valid.max(), 100)
        ax.plot(x_fit, slope * x_fit + intercept, linestyle='--', color='#e4542a', linewidth=2.5, zorder=4)
        ax.text(pos11, pos12, f"R²={r_val**2:.2f}, {format_p(p_val)}", transform=ax.transAxes, fontsize=10, color="#e03c31", ha="left", va="bottom", zorder=5)

    from matplotlib.lines import Line2D
    custom_legend = [
        Line2D([0], [0], color='gray', lw=5, label='Wet regions'),
        Line2D([0], [0], color='#e03c31', lw=5, label='Dry regions'),
    ]
    leg = plt.legend(handles=custom_legend, fontsize=10, frameon=False, loc='upper left', handlelength=1.0, handleheight=0.8)
    for line in leg.get_lines():
        line.set_linewidth(7)
    ax.set_xlim(*x_display_range)
    ax.set_xlabel("MAT (°C)", fontsize=14)
    ax.set_ylabel("EOS (DOY)", fontsize=14)
    ax.tick_params(labelsize=12)
    return ax, breakpoint_temp


Plot GIMMS


In [ ]:
import os
satellite = 'gimms'

veg_type = 0
df = read_satellite_data(veg_type, satellite)
df = df[
    (df['annual_t'] >= -20) & (df['annual_t'] <= 20) &
    (df['annual_p'] >= 0) & (df['annual_p'] <= 4)
]

min_z_count = 10
min_eos, max_eos = 250, 320
figa = plot_grid(df, 'annual_t', 'annual_p', 'eos', 'gimms EOS (snow filter)', (min_eos, max_eos), min_z_count, satellite)
os.makedirs(f"../../results/figure1/{satellite}", exist_ok=True)
figa.savefig(f"../../results/figure1/{satellite}/eos.png", dpi=500, bbox_inches='tight')
print(f"Saved to ../../results/figure1/{satellite}/eos.png")
